# **MODULO 1: Analissi de mercado musical para el artista**


* **Datos:**
    * Fuente de datos: Last.fm API (consumo real de usuarios)
    * Datos objetivo: Streams por país, Género musical, Playlist placement, TikTok trends, Crecimiento de artistas similares...
    * EXTRA DATOS: EDAD DE LA AUDIENCIA PARA AVERIGUAR TARGET SEGÚN PLATAFORMA. (MÉTRICAS AUDIENCIA TIK TOK)

* **Análisis (EDA):** crecimiento por género, países con más crecimiento, correlación entre playlist y streams, estacionalidad.
    * Feature Engineering: Ratio energía/valence, duración normalizada, explicit flag, features por artista

* **ML:** Predicción de streams de una canción, Predicción de viralidad, 
    * Modelos: Random Forest, XGBoost, LSTM para series temporales

* Output: “Tu canción tiene 65% probabilidad de viralizarse en México”, “Este beat funciona mejor en playlist de chill trap”





### **ÍNDICE DE IMPRESCINDIBLES PARA MODULO 1**



* **Módulo es el core estratégico: entender el mercado + base para predecir hits.**


* **DATOS: Se usan los datos de Last.fm API ya que  se basan en consumo real de usuarios.**

### Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
 


# **DATA**

## Imports

In [2]:
import time
import requests
import pandas as pd

## Recopilación de datos API

In [ ]:

API_KEY = '63e059c3c912a3f642daf2372484d183'
BASE_URL = 'http://ws.audioscrobbler.com/2.0/'
DELAY = 0.25  # 4 req/segundo < límite de 5/seg (margen de seguridad)


### Data base para el análisis:

* Función data TopTracks

In [17]:
def fetch_top_tracks(page=1, limit=200, retries=5):
    params = {
        'method': 'chart.getTopTracks',
        'api_key': API_KEY,
        'format': 'json',
        'page': page,
        'limit': limit
    }

    for attempt in range(retries):
        try:
            response = requests.get(BASE_URL, params=params,timeout=10)

            # 🔴 Rate limit
            if response.status_code == 429:
                wait_time = 2 ** attempt
                print(f"[429] Esperando {wait_time}s...")
                time.sleep(wait_time)
                continue

            # 🔴 Errores de servidor
            if response.status_code in [500, 502, 503]:
                wait_time = 2 ** attempt
                print(f"[{response.status_code}] Error servidor. Retry en {wait_time}s...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()
            return response.json()

        except requests.exceptions.RequestException as e:
            print(f"Error request: {e}")
            time.sleep(2 ** attempt)

    print(f"Fallo total en página {page}")
    return None

* Batch & DataFrame para TopTracks

In [18]:
all_tracks = []

for page in range(1, 201):

    data = fetch_top_tracks(page=page)

    if data is None:
        print(f"Error en página {page}")
        continue

    tracks = data['tracks']['track']
    all_tracks.extend(tracks)

    time.sleep(DELAY)

    if page % 50 == 0:
        temp_df = pd.DataFrame(all_tracks)
        temp_df.to_csv(f"backup_page_{page}.csv", index=False)

[502] Error servidor. Retry en 1s...
[502] Error servidor. Retry en 1s...
[502] Error servidor. Retry en 1s...
[502] Error servidor. Retry en 1s...
[502] Error servidor. Retry en 1s...
[502] Error servidor. Retry en 1s...
Error request: HTTPConnectionPool(host='ws.audioscrobbler.com', port=80): Read timed out. (read timeout=10)
Error request: HTTPConnectionPool(host='ws.audioscrobbler.com', port=80): Read timed out. (read timeout=10)
[502] Error servidor. Retry en 1s...
[502] Error servidor. Retry en 2s...
[502] Error servidor. Retry en 1s...
[502] Error servidor. Retry en 1s...
Error request: HTTPConnectionPool(host='ws.audioscrobbler.com', port=80): Read timed out. (read timeout=10)


KeyboardInterrupt: 

In [19]:
df = pd.DataFrame(all_tracks)
df.to_csv("backup_manual.csv", index=False)

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        9999 non-null   str   
 1   duration    9999 non-null   str   
 2   playcount   9999 non-null   str   
 3   listeners   9999 non-null   str   
 4   mbid        8096 non-null   str   
 5   url         9999 non-null   str   
 6   streamable  9999 non-null   object
 7   artist      9999 non-null   object
 8   image       9999 non-null   object
dtypes: object(3), str(6)
memory usage: 703.2+ KB


Errores:

| Código | Significado | Quién tiene el problema     |
| ------ | ----------- | --------------------------- |
| 403    | Forbidden   | ❌ tú (API key, permisos)    |
| 429    | Rate limit  | ⚠️ tú (demasiadas requests) |
| 502    | Bad Gateway | ❌ servidor (Last.fm)        |


### Data por países:

### Data por género musical:

### Data de la información del artista:

### Data de información de artistas similares:

### Data de la información del track

# **EDA**

1. **Análisis del mercado musical (descriptivo):** (1) Top canciones / artistas por periodo (tiempo); (2) Evolución temporal de popularidad; (3) Géneros en crecimiento vs decrecimiento

2. **Análisis por género:**

* Distribución de features por género: danceability, energy, valence, tempo

* ¿Qué hace único a cada género?

3. **Análisis geográfico** (1) Países con mayor consumo; (2) Diferencias culturales en features.

4. **Ciclos de éxito:** (1) Duración de popularidad de una canción; (2) Tiempo entre picos de streams; (3) Estacionalidad (ej: “hits de verano”)

**Features engineering (clave para predecir un hit):**


1. **Correlación con popularidad:** Qué variables impactan más: danceability ↑, energy ↑, valence (positivo vs triste)

* Heatmap de correlaciones

2. **Ingeniería de variables:** (1) Ratio energía/valence; (2) Duración normalizada; (3) Explicit vs no explicit; (4) Features agregadas por artista

3. **Segmentación de canciones:** 

* Clustering para KMeans
* Tipos de canciones: “club hit”, “chill”, “sad viral”,... 


### API Request

In [ ]:
ret = requests.get(f"https://ws.audioscrobbler.com/2.0/?method=chart.gettoptracks&api_key=63e059c3c912a3f642daf2372484d183&format=json")
ret.json()
for track in ret.json()["tracks"]["track"]:
  print(track["name"], ". Duracion:", track["duration"],"minutos")
music = pd.DataFrame(ret.json()["tracks"]["track"])
music.head()
# Cleaning the data frame:
music.drop(['mbid', 'image', 'url'],axis=1)


### Filtrando ds:

* Nombre y duracion de las canciones
* Comparación de la canción por titulo (happy or sad)
* Duración promedio de los top30 tracks


#### Nombre y duración de las canciones


In [ ]:
for track in ret.json()["tracks"]["track"]:
  print(track["name"], ". Duracion:", track["duration"],"minutos")


#### Comparación de grupos (canciones 'happy'-'sad'):

In [ ]:
df_happy = music[music['name'].str.contains('happy',case=False,na=False)]
df_happy.count()
df_sad = music[music['name'].str.contains('sad', case=False, na=False)]
df_sad.count()


#### Duración promedio de top30 tracks:

In [ ]:
df_top30_duration = music.sort_values(by='playcount', ascending=False) # .head(30)
df_top30_duration.iloc[:30]
df_top30_duration.head()

df_top30_duration.loc[:, 'duration'] = pd.to_numeric(df_top30_duration['duration'], errors='coerce')
mean = df_top30_duration['duration'].mean()
mean


### Procedencia de los artisitas top10:
 
* Oyentes de artists top10

**Info api request:**

In [ ]:
ret_artist = requests.get(f"https://ws.audioscrobbler.com/2.0/?method=geo.gettopartists&country=spain&api_key=63e059c3c912a3f642daf2372484d183&format=json")
ret_artist.json()
artists_country = pd.DataFrame(ret_artist.json()["topartists"]["artist"])
artists_country.head()


**Oyentes de los artistas top 10:**

In [3]:
df_top10 = music.head(10)
df_top10['duration'] = pd.to_numeric(df_top10['duration'], errors='coerce')
df_top10['listeners'] = pd.to_numeric(df_top10['listeners'], errors='coerce')

df_top10['artist_name'] = df_top10['artist'].apply(lambda x: x['name'])

plt.figure(figsize = (15, 5))

plt.bar(df_top10['artist_name'], df_top10['listeners'], color = ['skyblue'])
plt.xlabel('Artist name')
plt.ylabel('Listeners count')
plt.title("Listener of top 10 artist")
plt.show()


NameError: name 'music' is not defined

### Géneros musicales en tendencia global

In [ ]:

def get_trending_tags(api_key):
    url = f"https://ws.audioscrobbler.com/2.0/?method=tag.getTopTags&api_key={api_key}&format=json"
    response = requests.get(url)
    return response.json()['toptags']['tag']

api_key = '63e059c3c912a3f642daf2372484d183'
get_trending_tags(api_key)
ret_generes = get_trending_tags(api_key)
top_generes = pd.DataFrame(ret_generes)[['name', 'count', 'reach']]
top_generes.head()

# **ML**

1. **Definición del problema:** 

* Regresión → predecir popularidad (0–100)

* Clasificación → hit vs no hit

2. **Modelos básicos (mínimo viable):** (1) Linear Regression; (2) Random Forest; (3) XGBoost / LightGBM

3. **Evaluación:** (1) RMSE / MAE (regresión); (2) Accuracy / F1 (clasificación)

4. **Interpretabilidad:**

* Feature importance

* SHAP values (muy top para destacar)